## Tool definition

In [4]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [6]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [7]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [8]:
tool1.invoke({"x": 467})

21.61018278497431

## Adding to agents

In [9]:
import azure_openai_llm

llm = azure_openai_llm.create_azure_llm()

In [8]:
from langchain.agents import create_agent

agent = create_agent(llm, tools=[tool1],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [9]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The square root of 467 is approximately 21.61.


In [10]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='What is the square root of 467?', additional_kwargs={}, response_metadata={}, id='c1c9293d-bc0b-4cd4-b5d3-d1f3cd709cfc'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 74, 'total_tokens': 89, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-DFZuf80kCoHd28nIBInHyzTD1iwGa', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}

In [11]:
print(response["messages"][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_9GoxzqOBASIAhpVkCHSuhuSS', 'type': 'tool_call'}]


<H2>Manual Invocation</h2>

In [14]:
from pprint import pprint

llm_with_tools = llm.bind_tools(tools=[tool1])
query = "What is the square root of 900?"
response = llm_with_tools.invoke(query)
pprint(response.tool_calls)

[{'args': {'x': 900},
  'id': 'call_8MdRHhaYs5FNIu0QKMAcbj9Z',
  'name': 'square_root',
  'type': 'tool_call'}]


In [15]:
from langchain_core.messages import ToolMessage

# 1. Get the response from the LLM (which contains tool_calls)
query = "What is the square root of 1000?"
ai_msg = llm_with_tools.invoke(query)

for tool_call in ai_msg.tool_calls:
    # Look up the actual function from our 'tools' list
    # (In a real app, you'd use a dictionary mapping names to functions)
    selected_tool = {"square_root": square_root}[tool_call["name"].lower()]
    
    # 3. Execute the function with the arguments provided by the LLM
    tool_output = selected_tool.invoke(tool_call["args"])
    
    # 4. Create a ToolMessage to send the result back to the LLM
    # This must include the tool_call_id so the LLM knows which request this satisfies
    tool_message = ToolMessage(
        content=str(tool_output), 
        tool_call_id=tool_call["id"]
    )

# 5. Feed the result back to the LLM so it can give a final natural language answer
final_response = llm_with_tools.invoke([ai_msg, tool_message])
print(final_response.content)

The square root of 1000 is approximately 31.62.
